In [1]:
import warnings
from config import *
import os
import re
import pandas as pd
import numpy as np

# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [2]:
DA = "data/original data/h00da/H00CS_R.DA"
DCT = "data/original data/h00sta/H00CS_R.DCT"
randhrs = "data/original data/randhrs1992_2020v2.dta"
hrs_tracker = "data/original data/trk2022tr_r.dta"

In [3]:
dictionary = pd.read_excel('data/original data/h00dict.xlsx',header=None)
dictionary.columns = ['col num','col type','col name','col width','col label']
dictionary['col width'] = dictionary['col width'].apply(lambda x:int(float(x[1:-1])))
dictionary.head()

,col num,col type,col name,col width,col label
0,_column(1),str6,HHID,6,HOUSEHOLD IDENTIFIER
1,_column(7),str3,PN,3,PERSON NUMBER
2,_column(10),str1,GSUBHH,1,2000 SUB-HOUSEHOLD IDENTIFIER
3,_column(11),str1,FSUBHH,1,1998 SUB-HOUSEHOLD IDENTIFIER
4,_column(12),str3,GPN_SP,3,2000 SPOUSE/PARTNER PERSON NUMBER


In [4]:
df = pd.read_fwf(DA,widths=dictionary['col width'],names = dictionary['col label'])[['HOUSEHOLD IDENTIFIER','PERSON NUMBER','CS1C.PROXYIW-GOGN IMPAIRMT RATING']]
df.columns=['HHID','PN','PROXY']
df.head()

,HHID,PN,PROXY
0,2,10,NaN
1,3,10,NaN
2,3,20,NaN
3,10001,10,NaN
4,10003,30,NaN


In [5]:
# Time-invariant variables (from base interview)
timeinvariant_household_col = ['hidpn']
timeinvariant_respondent_col = ['gender', 'edyrs', 'bplace']
timeinvariant_col = [f'h{col}' for col in timeinvariant_household_col] + \
                    [f'ra{col}' for col in timeinvariant_respondent_col]

# Wave-specific variables for respondent and household
wave_household_col = ['child'] # Number of children
wave_respondent_col = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
    'agey_m',   #age
    'height',
    'weight',
    'smokev',   # Smoker status
    'proxy',    # Proxy interview indicator
    'imrc',  # immediate recall of a list of 10 words
    'dlrc',  # delayed recall of a list of 10 words
    'ser7',  # five trials of serial 7s
    'bwc20',  # backward counting
    'prmem',  # proxy rating of respondent memory
    'iadl5a',  # iadl5a = sum(phonea, moneya, medsa, shopa, mealsa), referred as "iadlza" in appendix
    'cendiv', # Census Division
    'mstat', # Marital Status
    'effort', # CESD Everything an effort
    'hibpe', # Ever had high blood pressure
    'diabe', # Ever had diabetes
    'cancre', # Ever had cancer
    'lunge', # Ever had lung disease
    'hearte', # Ever had heart problems
    'stroke', # Ever had stroke
    'arthre', # Ever had arthritis
    'vigact', # (For w5-w6): Whether vigorous phys act 3+/wk
    #'vgactx', # (For w7-w15) Whether vigorous phys act 3+/wk, Freq vigorous phys activ {finer scale}
    'slfmem', # Self-rated memory
    'livpar', # Number of living parents
    'momage', # Mother age current/at death
    'dadage', # Father age current/at death
    'livsib', # Number of living siblings
    'hlthlm', # Health problems limit work
    'hosp',	# Hospital stay(Overnight hospital stay since PrvIvw before death)
    'nrshom', # Nursing home stay (whether the Respondent reports any overnight nursing home stay in the reference period)
    'nrstim', # Nursing home stays
    'nrsnit', # Nights in nursing home (number of nights over all stays)
    'doctim', # Doctor visits (reported number of visits)
    'depres', # CESD Felt depressed
    'sleepr', # CESD Sleep was restless
    'whappy', # CESD Was happy
    'flone',    # CESD Felt lonely
    'fsad',     # CESD Felt sad
    'going',    # CESD Could not get going
    'enlife',   # CESD Enjoyed life
    'drink',    # Ever drinks any alcohol
    'smoken',   # Smokes now
    'toilta',   # Some Difficulty-Using the toilet
    'adl5a',    # Some Difficulty-IADLs /0-3
    'mapa',     # Some Difficulty-Use a map
    'walksa',   # Some Difficulty-Walk sev blocks
    'walk1a',   # Some Difficulty-Walk one block
    'sita',     # Some Difficulty-Sit for 2 hours
    'chaira',   # Some Difficulty-Get up fr chair
    'climsa',   # Some Difficulty-Clmb sev flt str
    'clim1a',   # Some Difficulty-Clmb 1 flt stair
    'stoopa',   # Some Difficulty-Stoop/Kneel/Crch
    'lifta',    # Some Difficulty-Lift/carry 10 lbs
    'dimea',    # Some Difficulty-Pick up a dime
    'armsa',    # Some Difficulty-Rch/xtnd arms up
    'pusha',    # Some Difficulty-Push/pull lg obj
    'mobila',   # Some Difficulty-Mobility /0-5
    'lgmusa',   # Some Diff-Large Muscle /0-4
    'grossa',   # Walk1/R,Clim1,Bed,Bath/0-5
    'finea',    # Dime/Eat/Dress /0-3
]

# Define wave and year based on wave number

# Set wave ( year = 2*(wave-5),e.g., 5 for year 2000)
wave = 5
year = 2 * (wave - 5)

# Construct final columns to load
columns = timeinvariant_col + \
          [f'h{wave}{col}' for col in wave_household_col] + \
          [f'r{wave}{col}' for col in wave_respondent_col]

# Read the selected columns from the dataset
chunks_main = pd.read_stata(randhrs, columns=columns)

# Map the correct 'prfin' column based on year
col_prfin = {0:'PROXY'}
original_prfin_col = col_prfin[year]
new_prfin_col = f'r{wave}prfin'

# Load prefin column dataset 
prfin_data = pd.read_csv("data/original data/h00cs_r.csv")

In [6]:
# Create unique person ID
#prfin_data['hhidpn'] = (prfin_data['HHID'] + prfin_data['PN']).astype(int)
prfin_data['hhidpn']=(prfin_data['HHID'].astype(str)+'0'+prfin_data['PN'].astype(str)).astype(int)

# Recode original prfin values into categories
prfin_data[new_prfin_col] = pd.cut(
    prfin_data[original_prfin_col],
    bins=[0, 1, 2, 3],
    labels=['1.none', '2.some', '3.prevented'],
    ordered=True
)

# Keep only hhidpn and recoded prfin variable
prfin_data = prfin_data[['hhidpn', new_prfin_col]]

# Merge with main dataset on hhidpn
chunks = pd.merge(
    chunks_main,
    prfin_data,
    on='hhidpn',
    how='left'
)

In [7]:
# Define new column name for race/ethnicity based on wave
raceeth_col = f'r{wave}raceeth'

# Load relevant columns from HRS tracker file
raceeth_tracker = pd.read_stata(
    hrs_tracker,
    columns=['HHID', 'PN', 'RACE', 'HISPANIC', 'NIWWAVE']
)

# Generate unique person ID
raceeth_tracker['hhidpn'] = (raceeth_tracker['HHID'] + raceeth_tracker['PN']).astype(int)

# Recode race/ethnicity following HRS coding logic:
# 0 = Non-Hispanic White
# 1 = Non-Hispanic Black
# 2 = Hispanic
# 3 = Non-Hispanic Other
raceeth_tracker.loc[raceeth_tracker['HISPANIC'].isin([1, 2, 3]), raceeth_col] = 2
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 1), raceeth_col] = 0
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 2), raceeth_col] = 1
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 7), raceeth_col] = 3

# Step 2: Filter for rows where NIWWAVE == 1
raceeth_tracker = raceeth_tracker[raceeth_tracker['NIWWAVE'] == 1]

# Drop original identifiers and raw race/ethnicity variables
raceeth_tracker.drop(columns=[ 'HISPANIC', 'RACE'], inplace=True)

# Check distribution of the recoded race/ethnicity variable
raceeth_tracker[raceeth_col].value_counts(dropna=False)

r5raceeth
0.0    14098
1.0     4146
2.0     2911
3.0      722
NaN       16
Name: count, dtype: int64

In [8]:
# Merge race/ethnicity tracker data into the main dataset using `hhidpn` as the key
chunks = pd.merge(
    chunks,                # main analysis dataset (already includes wave, prfin, etc.)
    raceeth_tracker,       # race/ethnicity tracker data
    on='hhidpn',           # merge key
    how='left'             # keep all cases from main dataset
)
chunks.head()

,hhidpn,ragender,raedyrs,rabplace,h5child,r5lbrf,r5shlt,r5agey_m,r5height,r5weight,r5smokev,r5proxy,r5imrc,r5dlrc,r5ser7,r5bwc20,r5prmem,r5iadl5a,r5cendiv,r5mstat,r5effort,r5hibpe,r5diabe,r5cancre,r5lunge,r5hearte,r5stroke,r5arthre,r5vigact,r5slfmem,r5livpar,r5momage,r5dadage,r5livsib,r5hlthlm,r5hosp,r5nrshom,r5nrstim,r5nrsnit,r5doctim,r5depres,r5sleepr,r5whappy,r5flone,r5fsad,r5going,r5enlife,r5drink,r5smoken,r5toilta,r5adl5a,r5mapa,r5walksa,r5walk1a,r5sita,r5chaira,r5climsa,r5clim1a,r5stoopa,r5lifta,r5dimea,r5armsa,r5pusha,r5mobila,r5lgmusa,r5grossa,r5finea,r5prfin,HHID,PN,NIWWAVE,r5raceeth
0,1010,1.male,16.0,2.mid atlantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,2.female,8.0,3.en central,9.0,5.retired,3.good,65.0,1.6002,44.45182,1.yes,0.not proxy,5.0,3.0,0.0,"2.correct, 1st try",NaN,0.0,3.en central,7.widowed,1.yes,0.no,0.no,0.no,1.yes,0.no,0.no,1.yes,0.no,2.very good,0.0,73.0,76.0,3.0,1.yes,0.no,0.no,0.0,0.0,1.0,0.no,0.no,1.yes,0.no,0.no,0.no,1.yes,1.yes,1.yes,0.no,0.0,NaN,1.yes,1.yes,0.no,0.no,1.yes,0.no,1.yes,1.yes,0.no,0.no,1.yes,3.0,2.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN
2,3010,1.male,12.0,9.pacific,5.0,4.partly retired,4.fair,64.0,1.6510,72.57440,0.no,0.not proxy,10.0,10.0,4.0,"2.correct, 1st try",NaN,0.0,9.pacific,1.married,0.no,0.no,0.no,0.no,0.no,1.yes,0.no,0.no,1.yes,4.fair,0.0,75.0,80.0,5.0,0.no,0.no,0.no,0.0,0.0,3.0,1.yes,0.no,1.yes,0.no,0.no,0.no,1.yes,1.yes,0.no,0.no,0.0,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.0,1.0,0.0,0.0,NaN,000003,010,1.0,0.0
3,3020,2.female,16.0,8.mountain,5.0,4.partly retired,3.good,61.0,1.6510,95.25390,0.no,0.not proxy,6.0,5.0,5.0,"2.correct, 1st try",NaN,0.0,9.pacific,1.married,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,1.yes,1.yes,3.good,0.0,65.0,78.0,0.0,1.yes,0.no,0.no,0.0,0.0,13.0,0.no,0.no,1.yes,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,0.0,0.no,0.no,0.no,0.no,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,0.no,0.0,1.0,0.0,0.0,NaN,000003,020,1.0,0.0
4,10001010,1.male,12.0,2.mid atlantic,0.0,5.retired,3.good,60.0,1.8288,79.37825,0.no,0.not proxy,7.0,7.0,5.0,"2.correct, 1st try",NaN,0.0,2.mid atlantic,8.never married,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,1.yes,3.good,0.0,79.0,66.0,1.0,1.yes,0.no,0.no,0.0,0.0,12.0,0.no,1.yes,1.yes,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,0.0,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.0,0.0,0.0,0.0,NaN,010001,010,1.0,0.0


In [9]:
# Convert time-invariant categorical variables to float (delete float setting)
timeinvariant_categorical_cols = ['ragender', 'raedyrs']

for col in timeinvariant_categorical_cols:
    chunks[col] = chunks[col].cat.codes.astype(float)
    chunks.loc[chunks[col] == -1, col] = float('nan')

# Recode one-hot indicator
for col in [
    'smokev', 'proxy', 'prmem', 'prfin','bwc20',  'effort',
    'hibpe', 'diabe', 'cancre', 'lunge', 'hearte', 'stroke',
    'arthre', 'vigact', 'slfmem', 'hlthlm', 'hosp', 'nrshom',
    'depres','sleepr', 'whappy', 'flone', 'fsad', 'going',
    'enlife', 'drink', 'smoken', 'toilta', 'mapa', 'walksa',
    'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa',
    'lifta', 'dimea', 'armsa','pusha'
]:

# delete categorical variables: 'cendiv', 'mstat', 'bplace' due to non-ordinal

    wave_col = f'r{wave}{col}'
    chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

# Bin education into stage groups
raedstg: ['1.0-7', '2.8-11', '3.12', '4.13+']
chunks['raedstg'] = pd.cut(
    chunks['raedyrs'],
    bins=[-1, 7, 11, 12, chunks['raedyrs'].max()],
    labels=['1.0-7', '2.8-11', '3.12', '4.13+']
)

# Convert all wave-specific categorical variables to float
wave_categorical_cols = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
]

for col in wave_categorical_cols:
    wave_col = f'r{wave}{col}'
    chunks[wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

/var/folders/hn/_0_krk556lb_wxhzwzg9sd8w0000gn/T/ipykernel_28485/2010597937.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.  1.  0. ... -1. -1. -1.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
/var/folders/hn/_0_krk556lb_wxhzwzg9sd8w0000gn/T/ipykernel_28485/2010597937.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.  0.  0. ... -1. -1. -1.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
/var/folders/hn/_0_krk556lb_wxhzwzg9sd8w0000gn/T/ipykernel_28485/2010597937.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ... 

In [10]:
# Drop the original 'raedyrs' column (already binned into 'raedstg' and encoded into 'redleq17')
chunks.drop(columns=['raedyrs'], inplace=True)

# (Optional) Reorder columns: Move key variables like hhidpn and race/ethnicity to the front
cols_order = ['hhidpn']
if f'r{wave}raceeth' in chunks.columns:
    cols_order.append(f'r{wave}raceeth')
if 'ragender' in chunks.columns:
    cols_order.append('ragender')
if 'raedstg' in chunks.columns:
    cols_order.append('raedstg')
if 'redleq17' in chunks.columns:
    cols_order.append('redleq17')

# Add remaining columns after key ones
cols_remaining = [col for col in chunks.columns if col not in cols_order]
chunks = chunks[cols_order + cols_remaining]
chunks['r5raceeth'].value_counts(dropna=False)

r5raceeth
NaN    20528
0.0    14098
1.0     4146
2.0     2911
3.0      722
Name: count, dtype: int64

In [11]:
# Define variable names
proxy_col = f'r{wave}proxy'
demscr_col = f'r{wave}demscr'
demcls_col = f'r{wave}demcls'

# Define column groups used to calculate cognitive score
self_dem_cols_wave = [f'r{wave}{col}' for col in SELF_DEM_COLS]
proxy_dem_cols_wave = [f'r{wave}{col}' for col in PROXY_DEM_COLS]

# Recode for self-respondents
self_mask = (chunks[proxy_col] == 0)
print(self_mask.sum())
chunks.loc[self_mask, demscr_col] = chunks.loc[self_mask, self_dem_cols_wave].sum(
    axis=1, min_count=len(self_dem_cols_wave)
)
chunks.loc[self_mask & (chunks[demscr_col] <= 6), demcls_col] = 1.0
chunks.loc[self_mask & (chunks[demscr_col] > 6), demcls_col] = 0.0

# Recode for proxy-respondents
proxy_mask = (chunks[proxy_col] == 1)
chunks.loc[proxy_mask, demscr_col] = chunks.loc[proxy_mask, proxy_dem_cols_wave].sum(
    axis=1, min_count=len(proxy_dem_cols_wave)
)
chunks.loc[proxy_mask & (chunks[demscr_col] >= 6), demcls_col] = 1.0
chunks.loc[proxy_mask & (chunks[demscr_col] < 6), demcls_col] = 0.0

# Drop rows with missing demscr (as they can't be classified)
# chunks.dropna(subset=[demscr_col], inplace=True)

# Drop raw demscr score now that demcls has been derived
chunks.drop(columns=[demscr_col], inplace=True)


17516


In [12]:
def remove_wave_prefix(col, wave):
    # Remove 'ra', 'r11', 'ha', 'h11' prefixes only
    return re.sub(rf'^(r|h)(a|{wave})', '', col)

# Apply clean renaming to all columns
chunks.rename(
    columns={col: remove_wave_prefix(col, wave) for col in chunks.columns},
    inplace=True
)

chunks['raceeth'].head()

0    NaN
1    NaN
2    0.0
3    0.0
4    0.0
Name: raceeth, dtype: float64

In [26]:
core_feature_cols = [
    'HHID', 'PN', 'hhidpn', 'NIWWAVE', # keys to merge, not x variables
    # all seleted x variables
    'edstg', "child", "lbrf", "shlt", "agey_m", "height", "weight", "smokev", "proxy", "cendiv", "mstat", "effort", "hibpe", "diabe", "cancre", "lunge", "hearte", "stroke", "arthre", "vigact", "slfmem", "livpar", "momage", "dadage", "livsib", "hlthlm", "hosp", "nrshom", "nrstim", "nrsnit", "doctim", "depres", "sleepr", "whappy", "flone", "fsad", "going", "enlife", "drink", "smoken",
    "toilta", "adl5a", "mapa", "walksa", "walk1a", "sita", "chaira", "climsa", "clim1a", "stoopa", "lifta", "dimea", "armsa",
    "pusha", "mobila", "lgmusa", "grossa", "finea", "raceeth","gender",
    "demcls" # y: Binary cognitive impairment classification
]

chunks_new=chunks[core_feature_cols]

# Keep only core features
chunks_new = chunks[core_feature_cols].copy()

# -----------------------------
# Summary before exclusion
# -----------------------------
n_total = len(chunks_new)

# age <= 50
n_age_le_50 = (chunks_new['agey_m'] <= 50).sum()

# demcls == 1
n_demcls_1 = (chunks_new['demcls'] == 1).sum()

# Make a filtered copy, dropping all rows with age ≤ 50, need to exclude all people (spouses or children) less than 50 age on baseline year to predict
chunks_new = chunks_new[chunks_new['agey_m'] > 50].copy().reset_index(drop=True)

# Create chunks_new by dropping all rows where demcls == 1, need to exclude all people who has dementia on baseline year to predict
chunks_new = chunks_new[chunks_new['demcls'] != 1].copy().reset_index(drop=True)

# Convert the 'raceeth' column to a categorical variable
chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')

chunks_new.head()

,HHID,PN,hhidpn,NIWWAVE,edstg,child,lbrf,shlt,agey_m,height,weight,smokev,proxy,cendiv,mstat,effort,hibpe,diabe,cancre,lunge,hearte,stroke,arthre,vigact,slfmem,livpar,momage,dadage,livsib,hlthlm,hosp,nrshom,nrstim,nrsnit,doctim,depres,sleepr,whappy,flone,fsad,going,enlife,drink,smoken,toilta,adl5a,mapa,walksa,walk1a,sita,chaira,climsa,clim1a,stoopa,lifta,dimea,armsa,pusha,mobila,lgmusa,grossa,finea,raceeth,gender,demcls
0,NaN,NaN,2010,NaN,2.8-11,9.0,4.0,2.0,65.0,1.6002,44.45182,1.0,0.0,3.en central,7.widowed,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,73.0,76.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,NaN,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0,NaN,1.0,0.0
1,000003,010,3010,1.0,3.12,5.0,3.0,3.0,64.0,1.6510,72.57440,0.0,0.0,9.pacific,1.married,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,3.0,0.0,75.0,80.0,5.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,000003,020,3020,1.0,4.13+,5.0,3.0,2.0,61.0,1.6510,95.25390,0.0,0.0,9.pacific,1.married,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,65.0,78.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,010001,010,10001010,1.0,3.12,0.0,4.0,2.0,60.0,1.8288,79.37825,0.0,0.0,2.mid atlantic,8.never married,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,79.0,66.0,1.0,1.0,0.0,0.0,0.0,0.0,12.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,010004,010,10004010,1.0,4.13+,2.0,4.0,2.0,60.0,1.8542,102.05775,1.0,0.0,2.mid atlantic,1.married,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,2.0,1.0,89.0,85.0,1.0,1.0,1.0,0.0,0.0,0.0,10.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Calculate and print the number of records (rows) in chunks_new
num_records = chunks_new.shape[0]
print(f"Number of 2000 year records in chunks_new: {num_records}")

Number of 2000 year records in chunks_new: 17359


In [15]:
# Rename the column 'vigact' to 'vgactx'
chunks_new = chunks_new.rename(columns={'vigact': 'vgactx'})

# Verify the change
print(chunks_new.columns)

Index(['HHID', 'PN', 'hhidpn', 'NIWWAVE', 'edstg', 'child', 'lbrf', 'shlt',
       'agey_m', 'height', 'weight', 'smokev', 'proxy', 'cendiv', 'mstat',
       'effort', 'hibpe', 'diabe', 'cancre', 'lunge', 'hearte', 'stroke',
       'arthre', 'vgactx', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib',
       'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres',
       'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink',
       'smoken', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita',
       'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa',
       'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'raceeth', 'gender',
       'demcls'],
      dtype='object')


In [17]:
#save data
chunks_new = pd.get_dummies(chunks_new,columns=chunks_new.select_dtypes(include=['category']).columns.tolist(),drop_first=True)
chunks_new = pd.get_dummies(chunks_new,columns=chunks_new.select_dtypes(include=['category']).columns.tolist(),drop_first=True)
chunks_new.to_csv('data/preprocessed data/2000.csv',na_rep="NA" ,index=False)

In [27]:
df_2000 = pd.read_csv('data/preprocessed data/2000.csv')
# proxy == 1
n_proxy_1 = (df_2000['proxy'] == 1).sum()
total_records = len(df_2000)
proxy_rate = n_proxy_1 /17336 

print(f"Number of people with proxy == 1: {n_proxy_1}")
print(f"Rate of proxy == 1: {proxy_rate:.2%}")
print(f"Number of people with age <= 50: {n_age_le_50}")
print(f"Number of people with demcls == 1: {n_demcls_1}")


Number of people with proxy == 1: 1292
Rate of proxy == 1: 7.45%
Number of people with age <= 50: 707
Number of people with demcls == 1: 1521


In [18]:
#Select 'demcls' and rename it to '__demcls' and HHID,PN, hhidpn
chunks_new.rename(columns={'demcls':'00demcls'}, inplace=True)
y = chunks_new[['HHID','PN', 'hhidpn' , '00demcls']]
y.to_csv('data/preprocessed data/_00y.csv',na_rep="NA" ,index=False)
y.head()

,HHID,PN,hhidpn,00demcls
0,NaN,NaN,2010,0.0
1,000003,010,3010,0.0
2,000003,020,3020,0.0
3,010001,010,10001010,0.0
4,010004,010,10004010,0.0
